In [1]:
# import sys
# !{sys.executable} -m pip install pandas selenium webdriver-manager openpyxl

In [2]:
import pandas as pd
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import UnexpectedAlertPresentException, WebDriverException
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import glob
import re
import time
import datetime
import os

In [3]:
def criar_driver():
    service = Service("/usr/bin/chromedriver")
    options = webdriver.ChromeOptions()
    options.binary_location = "/usr/bin/chromium-browser"
    options.add_argument("--start-maximized")
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return webdriver.Chrome(service=service, options=options)

In [4]:
PASTA_BACKUP = "/scraping/cnj_alvo/backup"
PASTA_SCRAPING = "/scraping/cnj_alvo/"

In [5]:
df_origem = pd.read_csv(f"{PASTA_SCRAPING}/df_cnj_webscraping.csv", dtype=str)
processos = df_origem['processo']

In [6]:
COLUNAS = ["processo", "data_distribuicao", "classe_judicial", "assunto", 
           "jurisdicao", "orgao_julgador", "status", "polo_ativo", "polo_passivo", "outros_interessados"]

In [7]:
def extrair_participantes(wait, xpath_table):
    participantes = []
    try:
        rows = wait.until(EC.presence_of_all_elements_located((By.XPATH, xpath_table)))
        
        for row in rows:
            try:
                name = row.find_element(By.XPATH, ".//td[1]//span[contains(@class,'text-bold') or contains(@class,'')]").text.strip()
                status_p = row.find_element(By.XPATH, ".//td[2]//div").text.strip()
                participantes.append(f"{name} [{status_p}]")
            except:
                continue
    except:
        return ""
        
    return ", ".join(participantes) if participantes else ""

In [8]:
def buscar_processo(driver, wait, processo_num):
    input_pesquisa = wait.until(EC.element_to_be_clickable(
        (By.ID, "fPP:numProcesso-inputNumeroProcessoDecoration:numProcesso-inputNumeroProcesso")
    ))

    try:
        input_pesquisa.clear()
        input_pesquisa.click()
    except Exception:
        driver.execute_script("arguments[0].click();", input_pesquisa)
        driver.execute_script("arguments[0].value = '';", input_pesquisa)

    time.sleep(0.5)
    input_pesquisa.send_keys(processo_num)
    time.sleep(0.5)

    btn_pesquisa = wait.until(EC.element_to_be_clickable((By.ID, "fPP:searchProcessos")))
    
    try:
        btn_pesquisa.click()
    except Exception:
        driver.execute_script("arguments[0].click();", btn_pesquisa)
    
    time.sleep(2)

In [9]:
def trocar_para_nova_aba(driver, wait):
    original = driver.current_window_handle
    wait.until(EC.number_of_windows_to_be(2))

    for handle in driver.window_handles:
        if handle != original:
            driver.switch_to.window(handle)
            return original

In [10]:
def extrair_dados_processo(wait):
    res = {}

    def buscar_texto(xpath):
        return wait.until(EC.presence_of_element_located((By.XPATH, xpath))).text.strip()

    res["data_distribuicao"] = buscar_texto("//label[contains(text(), 'Data da Distribuição')]/../../div[2]")
    res["classe_judicial"] = buscar_texto("//label[contains(text(), 'Classe Judicial')]/../../div[2]")
    res["assunto"] = buscar_texto("//label[contains(text(), 'Assunto')]/../../div[2]/div")
    res["jurisdicao"] = buscar_texto("//label[contains(text(), 'Jurisdição')]/../../div[2]")

    elemento_orgao = wait.until(
        EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'value') and .//b[text()='Órgão Julgador']]/div"))
    )
    res["orgao_julgador"] = elemento_orgao.text.replace("Órgão Julgador", "").strip()

    try:
        tb_status = wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//tbody[contains(@id, 'processoEvento')]/tr/td//span"))
        )
        res["status"] = tb_status[0].text.strip()
    except:
        res["status"] = "Sem movimentação"

    res["polo_ativo"] = extrair_participantes(wait, "//tbody[contains(@id, 'processoPartesPoloAtivoResumidoTableBinding')]/tr")
    res["polo_passivo"] = extrair_participantes(wait, "//tbody[contains(@id, 'processoPartesPoloPassivoResumidoTableBinding')]/tr")
    res["outros_interessados"] = extrair_participantes(wait, "//tbody[contains(@id, 'processoParteOutrosInteressadosResumidoTableBinding')]/tr")

    return res

In [11]:
def tentar_abrir_detalhes(driver):
    wait_curto = WebDriverWait(driver, 5)
    
    try:
        link = wait_curto.until(EC.presence_of_element_located(
            (By.XPATH, "//a[@title='Ver Detalhes']")
        ))
        driver.execute_script("arguments[0].click();", link)
        return True, None
    except:
        try:
            xpath_vazio = "//span[@class='rich-messages-label' and contains(text(), 'não encontrou nenhum processo')]"
            wait_curto.until(EC.presence_of_element_located((By.XPATH, xpath_vazio)))
            return False, "Não encontrado"
        except:
            print("❌ Erro: timeout ou bloqueio temporário")
            return False, "Erro: Timeout ou bloqueio temporário"

In [12]:
def processar_processo(driver, wait, processo_num):
    res = {col: "" for col in COLUNAS}
    res["processo"] = processo_num

    try:
        buscar_processo(driver, wait, processo_num)

        encontrou, status = tentar_abrir_detalhes(driver)
        if status:
            res["status"] = status
            return res

        original = trocar_para_nova_aba(driver, wait)

        dados = extrair_dados_processo(wait)
        res.update(dados)

        driver.close()
        driver.switch_to.window(original)

    except Exception as e:
        res["status"] = f"Erro: {type(e).__name__}"

        if len(driver.window_handles) > 1:
            try:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            except:
                pass

    return res

In [13]:
def processar_lote_registros():
    processos_pendentes = df_origem[df_origem['is_processed'] == 'False']
    
    if processos_pendentes.empty:
        print("🎉 Não há registros pendentes!")
        return

    driver = criar_driver()
    wait = WebDriverWait(driver, 20)
    
    # Arquivo de saída único por sessão, já na pasta certa
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    arquivo_saida = os.path.join(PASTA_BACKUP, f"trf1_cnj_{timestamp}.csv")
    
    # Cria o arquivo com cabeçalho
    pd.DataFrame(columns=COLUNAS).to_csv(arquivo_saida, index=False)
    print(f"🚀 Iniciando processamento. Restam {len(processos_pendentes)} processos.")
    print(f"📁 Salvando em: {arquivo_saida}")

    try:
        driver.get("https://pje1g.trf1.jus.br/consultapublica/ConsultaPublica/listView.seam")
        
        for index, row in processos_pendentes.iterrows():
            processo_num = row['processo']
            print(f"🔍 Processando {processo_num} (Índice: {index})...")

            try:
                res = processar_processo(driver, wait, processo_num)
                
                if "Erro:" in res.get("status", ""):
                    print(f"⚠️ Falha: {processo_num}: {res['status']}")
                    if "invalid session id" in res['status'].lower():
                        raise WebDriverException("Sessão expirada ou inválida.")
                    continue

                # Grava imediatamente no disco, sem acumular em memória
                pd.DataFrame([res]).to_csv(arquivo_saida, mode='a', header=False, index=False)
                df_origem.at[index, 'is_processed'] = 'True'
                df_origem.to_csv(f"{PASTA_SCRAPING}/df_cnj_webscraping.csv", index=False)
                print(f"✅ Sucesso: {processo_num} -> {res.get('status', '')}")
                
                time.sleep(random.uniform(1.5, 3.5))

            except UnexpectedAlertPresentException:
                print(f"🚨 Alerta detectado em {processo_num}. Seguindo...")
                try:
                    driver.switch_to.alert.accept()
                except:
                    pass
                continue

            except Exception as e:
                print(f"❌ Erro crítico em {processo_num}: {e}")
                break

    finally:
        try:
            driver.quit()
        except:
            pass
        print("🏁 Sessão encerrada.")

In [ ]:
processar_lote_registros()

🚀 Iniciando processamento. Restam 19831 processos.
📁 Salvando em: /home/analab/Documentos/laboratorios/python-data-science-ml/sumauma/scraping/cnj_alvo/backup/trf1_cnj_20260626_223157.csv
🔍 Processando 0000332-90.2019.4.01.3603 (Índice: 923)...
✅ Sucesso: 0000332-90.2019.4.01.3603 -> 08/11/2024 17:49:15 - Redistribuído por sorteio em razão de alteração de competência do órgão
🔍 Processando 0000333-09.2018.4.01.3604 (Índice: 924)...
✅ Sucesso: 0000333-09.2018.4.01.3604 -> 10/11/2025 13:30:03 - Processo Suspenso ou Sobrestado Por decisão judicial
🔍 Processando 0000333-67.2018.4.01.4102 (Índice: 925)...
✅ Sucesso: 0000333-67.2018.4.01.4102 -> 09/03/2026 13:23:11 - Processo devolvido à Secretaria
🔍 Processando 0000333-77.2016.4.01.3604 (Índice: 926)...
✅ Sucesso: 0000333-77.2016.4.01.3604 -> 03/06/2025 11:22:20 - Juntada de petição intercorrente
🔍 Processando 0000334-70.2017.4.01.3202 (Índice: 927)...
✅ Sucesso: 0000334-70.2017.4.01.3202 -> 07/05/2026 09:16:26 - Juntada de Certidão
🔍 Proce

In [ ]:
arquivos = glob.glob(f"{PASTA_BACKUP}/trf1_cnj_*.csv")
df_final = pd.concat([pd.read_csv(f, dtype=str) for f in arquivos], ignore_index=True)
df_final = df_final.drop_duplicates(subset=["processo"])
df_final.to_excel(f"{PASTA_SCRAPING}/cnj_trf1_scraping_acumulados.xlsx", index=False)
print("Arquivos unificados!")